---

In [ ]:
#!/usr/bin/env python3
"""CMC LRR Time Calculation"""

import pandas as pd
import numpy as np

# Load data
print("Loading data...")
df = pd.read_csv('proceed_radiomics_103_patients(two_years).csv')
print(f"Loaded {len(df)} patients\n")

# Combine local & regional recurrence dates
df["LRR_Date"] = df["Date.2"].fillna(df["Date.3"])

# Convert to datetime
df['Date - Finish'] = pd.to_datetime(df['Date - Finish'], errors='coerce')
df["LRR_Date"] = pd.to_datetime(df["LRR_Date"], errors='coerce')
df['Date.1'] = pd.to_datetime(df['Date.1'], errors='coerce')

# Calculate time in days
df['locoregional_recurrence_in_days'] = np.where(
    df['LRR_Date'].notna(),
    (df['LRR_Date'] - df['Date - Finish']).dt.days,
    (df['Date.1'] - df['Date - Finish']).dt.days
)

# Create event indicator
df['LRR_event'] = df['LRR_Date'].notna().astype(int)

# Check for negative values
negative = df[df['locoregional_recurrence_in_days'] < 0]

if len(negative) > 0:
    print(f"⚠️  Found {len(negative)} patients with negative time values:")
    print("="*60)
    for idx, row in negative.iterrows():
        patient_id = row.get('id', f'Row {idx}')
        print(f"\nPatient {patient_id}:")
        print(f"  Treatment Finish: {row['Date - Finish']}")
        if pd.notna(row['LRR_Date']):
            print(f"  LRR Date: {row['LRR_Date']}")
        else:
            print(f"  Follow-up: {row['Date.1']}")
        print(f"  Time: {row['locoregional_recurrence_in_days']} days (NEGATIVE)")
    print(f"\n{'='*60}")
    print(f"These {len(negative)} patients excluded from output.\n")

# Remove negative values
df_clean = df[df['locoregional_recurrence_in_days'] >= 0].copy()

# Save cleaned data
output_file = 'cmc_data_with_lrr_days.csv'
df_clean.to_csv(output_file, index=False)

print(f"Saved: {output_file}")
print(f"  Total: {len(df_clean)} patients")
print(f"  LRR: {df_clean['LRR_event'].sum()} patients")
print(f"  Non-LRR: {(df_clean['LRR_event'] == 0).sum()} patients")

# Quick stats
lrr = df_clean[df_clean['LRR_event'] == 1]['locoregional_recurrence_in_days']
no_lrr = df_clean[df_clean['LRR_event'] == 0]['locoregional_recurrence_in_days']

print(f"\nMedian time to LRR: {lrr.median():.0f} days")
print(f"Median follow-up (non-LRR): {no_lrr.median():.0f} days")
print("\n✓ Done!")

---